# Autoformalization Evaluation

In [ ]:
import sys, re, spacy
sys.path.insert(0, '/home/flopezp/LogicSim')  
import pandas as pd
from logicsim import utils, metrics

dataset_path = '/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{}/{}_all_samples.csv'
model_list = utils.model_list

for elem in model_list:
    model_id = elem[0].split('/')[1]
    # IDEAL USE:
    # metrics.logicsimautoform(model_id)
    # Y QUE SE EVALÚE TODO

In [ ]:
dataset_path = '/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{}/{}_all_samples.csv'
model_list = utils.model_list
ex_data = pd.read_csv(dataset_path.format('FOLIO', model_list[0][0].split('/')[1]))
ex_data = ex_data.drop(columns=['prompt_index', 'sample_index', 'prompt_text'])
ex_data.head()

In [ ]:
clean_text = []
none_regex = 0
avg_ans_length = 0
answer_len = len(ex_data['generated_text'].to_list())

for elem in ex_data['generated_text'].to_list():
    avg_ans_length += len(elem)
    text_split = elem.split('<text>')
    # Tal vez no sea search la mejor opción.
    # Podemos evaluar distintas formas de extraer la proposición final usando regex.
    regex_extraction = re.search(r'(<text>)[A-z0-9∀∃\n⊕→¬∧ \t()"á,∨]+(<\/text>)', elem)

    #Second regex
    # regex2_extraction = re.search(r'<text>[\\Śą<>≤A-z0-9:á ∀∧→⊕¬←∨∃↔∈()’\'=≠?.\-,\n"]+<\/text>', elem) 
      
    if regex_extraction != None:
        texto = regex_extraction.group()[6:-7]
        cleaned = re.sub('  ', '', texto)
        clean_text.append(cleaned)
    else:
        none_regex += 1
        clean_text.append(None)

filtered_ans_len = 0
for elem in clean_text:
    if elem != None:
        filtered_ans_len += len(elem)


print(f'Longitud promedio de respuesta: {round(avg_ans_length/answer_len, 4)}')
print(f'Valores totales: {answer_len}')
print(f'Valores mal generados: {none_regex}. Porcentaje: {round(none_regex/answer_len, 4)*100}%')
print(f'Longitud promedio de respuesta filtrada: {round(filtered_ans_len/(answer_len - none_regex), 4)}')

In [ ]:
folio_val = pd.read_json(r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
folio_premises = folio_val['premises-FOL'].to_list()

# ----------------------------------------------------------------------------------------------------------
# Con esto ya podemos medir la primeras dos partes de la métrica: PARSING & CARDINALITY_EQUALITY
# ----------------------------------------------------------------------------------------------------------

llm_parse = False
ds_parse = False
cardinal_list = []

parsing_pairs = 0

for i in range(int(len(clean_text)/5)):
    ds_value = folio_premises[i]
    nones = 0
    for j in range(5):
        current_index = i*5 + j
        print('-'*50)
        print('\t Evaluating instance ', current_index)
        print('-'*50)
        if clean_text[current_index] == None:
            nones += 1
        else:
            ds_parse = metrics.lark_based_parsability(ds_value, parser = metrics.parser)
            llm_parse = metrics.lark_based_parsability(clean_text[current_index], parser = metrics.parser)
            print('-'*20, 'Parsing', '-'*20)
            print(f'DS Parses: {ds_parse}')
            print(f'LLM Autoform Parses: {llm_parse}')
            if llm_parse and ds_parse:
                parsing_pairs += 1
                try:
                    print('-'*20, 'Cardinality', '-'*20)
                    cardinal_equality = metrics.verify_cardinality(ds_value, clean_text[current_index])
                    print(f'Cardinality Equality: {cardinal_equality[0]}')
                    print(f'Constant Errors: {cardinal_equality[1]}')
                    print(f'Predicate Errors: {cardinal_equality[2]}')
                    if cardinal_equality[0] == True:
                        cardinal_list.append((i, current_index))
                except:
                    print('-'*20, 'Cardinality', '-'*20)
                    print(f'Cardinality Equality: False')
                    print(f'DS value: {ds_value}')
                    print(f'LLM value: {clean_text[current_index]}')

    print('-'*25)


print(f'Cantidad de instancias que parsean: {parsing_pairs}')
print(f'Cardinal equal values: {len(cardinal_list)}')

In [ ]:
for elem in cardinal_list:
    print('='* 40)
    print(f'\t Índice: {elem}')
    gold = folio_premises[elem[0]]
    llm = clean_text[elem[1]]
    query = folio_val['conclusion-FOL'].iloc[elem[0]]
    gold_ag, llm_ag, query_ag = metrics.name_agnostic_transformation(gold, llm, query)
    print(f'Gold Name Agnostic: \n {gold_ag}')
    print('-'*15)
    print(f'LLM Name Agnostic: {llm_ag}')
    print('-'*15)
    print(f'Queries Name Agnostic: \n {query_ag}')